# 03 — Narrative Signals (Signal B)

Builds narrative risk signals from `Report 1 | Narrative` using:
1) zero-shot multi-label classification (core) — run via OpenRouter `deepseek/deepseek-v4-flash`
2) lightweight unsupervised discovery pass (TF-IDF + KMeans)

Method follows `production/spec/P03_methodology.md` §6. The §6.1 *pilot* used local GLiNER2; the realized full run uses the hosted model above (see that note for the pivot). Output files keep the `gliner_label_*` name for historical continuity.

In [ ]:
# provenance: primary=OP48 authors=[OP48, CDX53] id=P0009 ts=2026-06-19T02:12+03:00
from __future__ import annotations

from pathlib import Path
import os
import json
import time
import math
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import httpx

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

ROOT = Path.cwd()
if not (ROOT / "data" / "processed" / "asrs.parquet").exists():
    ROOT = ROOT.parent.parent

DATA_PATH = ROOT / "data" / "processed" / "asrs.parquet"
OUT_DIR = ROOT / "production" / "output" / "narrative_signals"
PLOT_DIR = OUT_DIR / "plots"
OUT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

WINDOW_START = 201101
WINDOW_END = 202512
TRAILING_EXCLUDE_MONTHS = 2
BASELINE_MONTHS = 24
WATCH_Z = 2.0
STRONG_Z = 3.0
PERSISTENCE_MONTHS = 3

# Full run unless explicitly capped.
MAX_NARRATIVES = None
CLS_THRESHOLD = 0.40

# OpenRouter + DeepSeek V4 Flash execution controls.
OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"
OR_CONCURRENCY = 48
OR_BATCH_SIZE = 12   # narratives/request
OR_MAX_RETRIES = 6
OR_TIMEOUT_S = 120

RISK_LABELS = [
    "runway incursion / ground conflict / surface movement",
    "ATC staffing or workload pressure",
    "pilot fatigue",
    "automation mode confusion",
    "GPS interference or jamming",
    "UAS or drone encounter",
    "wake turbulence encounter",
    "laser illumination event",
    "unstabilized approach",
    "smoke fire fumes odor",
]


def load_env_file(path: Path):
    if not path.exists():
        return
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        k = k.strip()
        v = v.strip().strip('"').strip("'")
        os.environ.setdefault(k, v)


load_env_file(ROOT / ".env")
OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "").strip()
OPENROUTER_MODEL = os.environ.get("OPENROUTER_MODEL", "deepseek/deepseek-v4-flash").strip()

if not OPENROUTER_API_KEY:
    raise RuntimeError("OPENROUTER_API_KEY not found in environment or .env")

print(f"DATA_PATH={DATA_PATH}")
print(f"OUT_DIR={OUT_DIR}")
print(f"OPENROUTER_MODEL={OPENROUTER_MODEL}")

DATA_PATH=data/processed/asrs.parquet
OUT_DIR=production/output/narrative_signals
OPENROUTER_MODEL=deepseek/deepseek-v4-flash


In [2]:
def rolling_mad(values: np.ndarray) -> float:
    med = np.median(values)
    return float(np.median(np.abs(values - med)))


def longest_run(mask: pd.Series) -> int:
    run = 0
    best = 0
    for v in mask.fillna(False).astype(bool):
        if v:
            run += 1
            best = max(best, run)
        else:
            run = 0
    return best


def first_run_ym(mask: pd.Series, yms: pd.Series, min_run: int):
    run = 0
    for ok, ym in zip(mask.fillna(False).astype(bool), yms):
        if ok:
            run += 1
            if run >= min_run:
                return int(ym)
        else:
            run = 0
    return np.nan


def normalize_narrative(text: str) -> str:
    if not isinstance(text, str):
        return ""
    t = text.replace("ZZZ", " ").replace("[date]", " ")
    return " ".join(t.split())


def chunked(seq, size: int):
    for i in range(0, len(seq), size):
        yield seq[i:i + size]


def parse_llm_json(content: str) -> dict:
    content = content.strip()
    if content.startswith("```"):
        content = content.strip("`")
        if content.startswith("json"):
            content = content[4:]
    return json.loads(content)


def build_prompt(batch_records: list[dict]) -> tuple[str, str]:
    labels_json = json.dumps(RISK_LABELS)
    items = "\n".join(
        [f"{i}. ym={r['ym']}\n{r['narrative']}" for i, r in enumerate(batch_records)]
    )
    system = (
        "You are a strict multi-label classifier for aviation safety narratives. "
        "Return ONLY valid JSON, no markdown."
    )
    user = f"""
Classify each narrative into zero or more labels from this fixed set:
{labels_json}

Rules:
- Multi-label allowed.
- Only output labels from the fixed set.
- Use confidence in [0,1].
- If no label applies, output empty labels array.
- Keep order by idx.

Return JSON exactly with shape:
{{"items":[{{"idx":0,"labels":[{{"label":"...","confidence":0.0}}]}}]}}

Narratives:
{items}
""".strip()
    return system, user


def classify_batch_openrouter(client: httpx.Client, batch_records: list[dict]) -> list[dict]:
    system, user = build_prompt(batch_records)
    payload = {
        "model": OPENROUTER_MODEL,
        "messages": [
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
        "temperature": 0,
        "response_format": {"type": "json_object"},
    }

    backoff = 1.0
    for attempt in range(OR_MAX_RETRIES):
        try:
            r = client.post(OPENROUTER_URL, json=payload, timeout=OR_TIMEOUT_S)
            if r.status_code in (429, 500, 502, 503, 504):
                raise RuntimeError(f"retryable_status:{r.status_code}")
            r.raise_for_status()
            data = r.json()
            content = data["choices"][0]["message"]["content"]
            parsed = parse_llm_json(content)
            items = parsed.get("items", []) if isinstance(parsed, dict) else []
            out = []
            for i, rec in enumerate(batch_records):
                labels = []
                confs = []
                if i < len(items) and isinstance(items[i], dict):
                    for it in items[i].get("labels", []):
                        if isinstance(it, dict):
                            lbl = str(it.get("label", "")).strip()
                            conf = float(it.get("confidence", np.nan)) if it.get("confidence") is not None else np.nan
                            if lbl in RISK_LABELS and (np.isnan(conf) or conf >= CLS_THRESHOLD):
                                labels.append(lbl)
                                confs.append(conf)
                out.append(
                    {
                        "ACN": rec["ACN"],
                        "ym": int(rec["ym"]),
                        "labels": labels,
                        "confs": confs,
                    }
                )
            return out
        except Exception:
            if attempt == OR_MAX_RETRIES - 1:
                return [
                    {"ACN": rec["ACN"], "ym": int(rec["ym"]), "labels": [], "confs": []}
                    for rec in batch_records
                ]
            time.sleep(backoff)
            backoff *= 1.8

In [3]:
# Load narrative base.
base = (
    pl.read_parquet(DATA_PATH)
    .select(["ACN", "Date", "Report 1 | Narrative"])
    .with_columns(pl.col("Date").cast(pl.Int64).alias("ym"))
    .filter((pl.col("ym") >= WINDOW_START) & (pl.col("ym") <= WINDOW_END))
    .rename({"Report 1 | Narrative": "narrative"})
    .to_pandas()
)
base["narrative"] = base["narrative"].fillna("").map(normalize_narrative)
base = base[base["narrative"].str.len() > 20].copy()
base["year"] = base["ym"] // 100

if MAX_NARRATIVES and len(base) > MAX_NARRATIVES:
    sampled = []
    for _, g in base.groupby("year"):
        frac = MAX_NARRATIVES / len(base)
        n = max(1, int(round(len(g) * frac)))
        sampled.append(g.sample(n=min(n, len(g)), random_state=42))
    work = pd.concat(sampled, ignore_index=True)
else:
    work = base.copy()

work = work.sort_values("ym").reset_index(drop=True)
print(f"narratives total={len(base):,}  work={len(work):,}")
work[["ACN", "ym", "narrative"]].head(3)

narratives total=79,572  work=79,572


,ACN,ym,narrative
0,925960,201101,After a normal engine start; run-up; taxi and ...
1,925957,201101,Upon receiving clearance on the ground; Cleara...
2,925838,201101,After the flight we assisted the Flight Attend...


In [4]:
# OpenRouter (DeepSeek V4 Flash) multi-label classification + signal scoring.
headers = {
    "Authorization": f"Bearer {OPENROUTER_API_KEY}",
    "Content-Type": "application/json",
    "HTTP-Referer": "https://local.iata-case-study",
    "X-Title": "IATA-Case-Study-SignalB",
}

records = work[["ACN", "ym", "narrative"]].to_dict("records")
batches = list(chunked(records, OR_BATCH_SIZE))
print(f"batches={len(batches):,}, concurrency={OR_CONCURRENCY}, batch_size={OR_BATCH_SIZE}")

results = []
start = time.time()
with httpx.Client(headers=headers, timeout=OR_TIMEOUT_S) as client:
    with ThreadPoolExecutor(max_workers=OR_CONCURRENCY) as ex:
        futures = {ex.submit(classify_batch_openrouter, client, b): i for i, b in enumerate(batches)}
        done = 0
        for fut in as_completed(futures):
            out = fut.result()
            results.extend(out)
            done += 1
            if done % 100 == 0 or done == len(batches):
                elapsed = time.time() - start
                rate = done / elapsed if elapsed > 0 else 0
                print(f"completed {done}/{len(batches)} batches ({rate:.2f} batch/s)")

rows = []
for r in results:
    if not r["labels"]:
        rows.append({"ACN": r["ACN"], "ym": int(r["ym"]), "risk_label": None, "confidence": np.nan})
    else:
        for lbl, conf in zip(r["labels"], r["confs"] if r["confs"] else [np.nan] * len(r["labels"])):
            rows.append({"ACN": r["ACN"], "ym": int(r["ym"]), "risk_label": lbl, "confidence": conf})

assign = pd.DataFrame(rows)
assign.to_parquet(OUT_DIR / "gliner_label_assignments.parquet", index=False)

totals = work.groupby("ym", as_index=False).size().rename(columns={"size": "total_reports"})
label_counts = (
    assign.dropna(subset=["risk_label"]) 
    .groupby(["risk_label", "ym"], as_index=False)
    .size().rename(columns={"size": "count"})
)

if label_counts.empty:
    raise RuntimeError("No labels were returned by OpenRouter classification. Check key/model/rate limits.")

label_series = label_counts.merge(totals, on="ym", how="left")
label_series["share"] = label_series["count"] / label_series["total_reports"]

# Build full monthly grid per label.
all_ym = pd.DataFrame({"ym": sorted(work["ym"].unique())})
all_labels = pd.DataFrame({"risk_label": sorted(label_counts["risk_label"].unique())})
full = all_labels.merge(all_ym, how="cross").merge(label_series, on=["risk_label", "ym"], how="left")
full["count"] = full["count"].fillna(0).astype(int)
full = full.merge(totals, on="ym", how="left", suffixes=("", "_tot"))
full["total_reports"] = full["total_reports_tot"].fillna(full["total_reports"]).astype(int)
full = full.drop(columns=[c for c in ["total_reports_tot"] if c in full.columns])
full["share"] = np.where(full["total_reports"] > 0, full["count"] / full["total_reports"], np.nan)
full["date"] = pd.to_datetime(full["ym"].astype(str), format="%Y%m")

trailing = set(sorted(full["ym"].unique())[-TRAILING_EXCLUDE_MONTHS:])

scored_frames = []
cand_rows = []
for lbl, g in full.groupby("risk_label", sort=False):
    g = g.sort_values("ym").copy()
    base_share = g["share"].shift(1)
    g["median_24"] = base_share.rolling(BASELINE_MONTHS, min_periods=BASELINE_MONTHS).median()
    g["mad_24"] = base_share.rolling(BASELINE_MONTHS, min_periods=BASELINE_MONTHS).apply(rolling_mad, raw=True)
    denom = 1.4826 * g["mad_24"]
    g["z"] = np.where(denom > 0, (g["share"] - g["median_24"]) / denom, np.nan)
    g["z"] = g["z"].replace([np.inf, -np.inf], np.nan)
    g["yoy_share_delta"] = g["share"] - g["share"].shift(12)
    g["is_trailing_excluded"] = g["ym"].isin(trailing)

    baseline_ready = g["median_24"].notna() & g["mad_24"].notna()
    common_fire = baseline_ready & (~g["is_trailing_excluded"]) & (g["yoy_share_delta"] > 0)
    g["watch_fire"] = common_fire & (g["z"] >= WATCH_Z)
    g["strong_fire"] = common_fire & (g["z"] >= STRONG_Z)

    first_watch = first_run_ym(g["watch_fire"], g["ym"], PERSISTENCE_MONTHS)
    first_strong = first_run_ym(g["strong_fire"], g["ym"], PERSISTENCE_MONTHS)
    watch_run = longest_run(g["watch_fire"])
    strong_run = longest_run(g["strong_fire"])

    if not pd.isna(first_strong):
        first_fire = int(first_strong); fire_level = "strong"; sustained = strong_run
    elif not pd.isna(first_watch):
        first_fire = int(first_watch); fire_level = "watch"; sustained = watch_run
    else:
        first_fire = np.nan; fire_level = "none"; sustained = 0

    latest = g.iloc[-1]
    peak_z = float(g.loc[g["watch_fire"], "z"].max()) if g["watch_fire"].any() else np.nan
    rank = (peak_z * sustained) if pd.notna(peak_z) else 0.0

    cand_rows.append({
        "risk_label": lbl,
        "latest_share": float(latest["share"]) if pd.notna(latest["share"]) else np.nan,
        "z_latest": float(latest["z"]) if pd.notna(latest["z"]) else np.nan,
        "yoy_share_delta": float(latest["yoy_share_delta"]) if pd.notna(latest["yoy_share_delta"]) else np.nan,
        "first_fire_ym": first_fire,
        "months_sustained": int(sustained),
        "peak_z": peak_z,
        "rank": rank,
        "fire_level": fire_level,
    })
    scored_frames.append(g)

cand = pd.DataFrame(cand_rows).sort_values(["rank", "peak_z"], ascending=False).reset_index(drop=True)
scored = pd.concat(scored_frames, ignore_index=True)

cand.to_csv(OUT_DIR / "narrative_signal_candidates.csv", index=False)
scored.to_parquet(OUT_DIR / "gliner_label_monthly_share.parquet", index=False)

# Plot top 6 labels with persisted fire.
top = cand[cand["first_fire_ym"].notna()].head(6)
for _, row in top.iterrows():
    lbl = row["risk_label"]
    g = scored[scored["risk_label"] == lbl].sort_values("ym")
    fig, ax = plt.subplots(figsize=(11.5, 4.2))
    ax.plot(g["date"], g["share"], label="Share", linewidth=1.6)
    ax.plot(g["date"], g["median_24"], label="24m baseline", linewidth=1.1)
    band = 1.4826 * g["mad_24"]
    ax.fill_between(g["date"], g["median_24"] - band, g["median_24"] + band, alpha=0.15, label="±1 MAD")
    wf = g[g["watch_fire"]]
    sf = g[g["strong_fire"]]
    if not wf.empty:
        ax.scatter(wf["date"], wf["share"], s=16, label="watch")
    if not sf.empty:
        ax.scatter(sf["date"], sf["share"], s=20, label="strong")
    tr = g[g["is_trailing_excluded"]]
    if not tr.empty:
        ax.axvspan(tr["date"].min(), tr["date"].max() + pd.offsets.MonthEnd(1), color="gray", alpha=0.12)
    ax.set_title(lbl)
    ax.set_xlabel("Year")
    ax.set_ylabel("Narrative label share")
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.tick_params(axis="x", rotation=0)
    ax.legend(fontsize=8)
    fig.tight_layout()
    safe = "".join(ch if ch.isalnum() else "_" for ch in lbl)[:90]
    fig.savefig(PLOT_DIR / f"label_{safe}.png", dpi=140)
    plt.close(fig)

print("wrote", OUT_DIR / "narrative_signal_candidates.csv")
print("wrote", OUT_DIR / "gliner_label_monthly_share.parquet")
print("plots", len(list(PLOT_DIR.glob("label_*.png"))))
cand.head(15)

batches=6,631, concurrency=48, batch_size=12


completed 100/6631 batches (2.61 batch/s)


completed 200/6631 batches (2.51 batch/s)


completed 300/6631 batches (2.58 batch/s)


completed 400/6631 batches (2.70 batch/s)


...

completed 6500/6631 batches (3.10 batch/s)


completed 6600/6631 batches (3.10 batch/s)


completed 6631/6631 batches (2.99 batch/s)


wrote production/output/narrative_signals/narrative_signal_candidates.csv
wrote production/output/narrative_signals/gliner_label_monthly_share.parquet
plots 5


,risk_label,latest_share,z_latest,yoy_share_delta,first_fire_ym,months_sustained,peak_z,rank,fire_level
0,GPS interference or jamming,0.029787,0.146121,-0.017383,201905.0,5,27.006585,135.032924,watch
1,smoke fire fumes odor,0.074468,0.401607,0.008430,201905.0,4,5.346051,21.384202,watch
2,UAS or drone encounter,0.040426,0.210828,-0.002027,201407.0,3,6.220995,18.662985,watch
3,runway incursion / ground conflict / surface m...,0.189362,-0.986588,0.007758,202106.0,4,4.244439,16.977757,watch
4,unstabilized approach,0.074468,-0.786208,-0.010438,201402.0,3,4.756831,14.270493,strong
5,laser illumination event,0.004255,NaN,0.004255,NaN,0,5.143550,0.000000,none
6,wake turbulence encounter,0.012766,-2.183366,-0.024970,NaN,0,4.888362,0.000000,none
7,pilot fatigue,0.025532,1.133746,0.006664,NaN,0,3.999803,0.000000,none
8,ATC staffing or workload pressure,0.046809,0.091195,0.009073,NaN,0,3.033944,0.000000,none
9,automation mode confusion,0.076596,1.090676,0.003483,NaN,0,2.547076,0.000000,none


In [5]:
# Lightweight unsupervised discovery pass (sampled) for unlabeled emergents.
DISCOVERY_SAMPLE = min(20000, len(work))
TOPIC_K = 16

disc = work.sample(n=DISCOVERY_SAMPLE, random_state=42).copy() if len(work) > DISCOVERY_SAMPLE else work.copy()

vec = TfidfVectorizer(
    max_features=8000,
    ngram_range=(1, 2),
    min_df=10,
    max_df=0.6,
    stop_words="english",
)
X = vec.fit_transform(disc["narrative"])

km = KMeans(n_clusters=TOPIC_K, random_state=42, n_init="auto")
disc["topic_id"] = km.fit_predict(X)

terms = np.array(vec.get_feature_names_out())
centers = km.cluster_centers_
rows = []
for i in range(TOPIC_K):
    top_idx = centers[i].argsort()[-12:][::-1]
    rows.append({"topic_id": i, "top_terms": ", ".join(terms[top_idx])})

topic_terms = pd.DataFrame(rows)
topic_terms.to_csv(OUT_DIR / "topic_discovery_top_terms.csv", index=False)

topic_month = disc.groupby(["topic_id", "ym"], as_index=False).size().rename(columns={"size": "count"})
month_tot = disc.groupby("ym", as_index=False).size().rename(columns={"size": "total_reports"})
topic_month = topic_month.merge(month_tot, on="ym", how="left")
topic_month["share"] = topic_month["count"] / topic_month["total_reports"]

# Rising topics by recent-vs-early mean share.
ranks = []
for t, g in topic_month.groupby("topic_id"):
    g = g.sort_values("ym")
    early = g.head(max(1, len(g)//3))["share"].mean()
    late = g.tail(max(1, len(g)//3))["share"].mean()
    ranks.append({"topic_id": t, "early_share": early, "late_share": late, "growth": late - early})

topic_rank = pd.DataFrame(ranks).merge(topic_terms, on="topic_id", how="left").sort_values("growth", ascending=False)
topic_rank.to_csv(OUT_DIR / "topic_discovery_growth.csv", index=False)

summary = [
    "# Signal B summary\n",
    f"- narratives used for zero-shot classification (OpenRouter DeepSeek V4 Flash): **{len(work):,}** (from total {len(base):,})\n",
    f"- labels tracked: **{len(RISK_LABELS)}**\n",
    f"- labels with persisted fire: **{int(cand['first_fire_ym'].notna().sum())}**\n",
    f"- discovery sample size: **{len(disc):,}**, topics: **{TOPIC_K}**\n",
    "\n## Top narrative labels\n",
]
for _, r in cand.head(6).iterrows():
    summary.append(
        f"- `{r['risk_label']}` | first fire `{int(r['first_fire_ym']) if pd.notna(r['first_fire_ym']) else 'none'}` | "
        f"level `{r['fire_level']}` | peak z `{r['peak_z'] if pd.notna(r['peak_z']) else float('nan'):.2f}` | rank `{r['rank']:.2f}`\n"
    )
summary.append("\n## Top discovery topics (growth)\n")
for _, r in topic_rank.head(5).iterrows():
    summary.append(f"- topic {int(r['topic_id'])}: {r['top_terms']}\n")

(OUT_DIR / "P04_narrative_summary.md").write_text("".join(summary), encoding="utf-8")
print("wrote", OUT_DIR / "topic_discovery_top_terms.csv")
print("wrote", OUT_DIR / "topic_discovery_growth.csv")
print("wrote", OUT_DIR / "P04_narrative_summary.md")

topic_rank.head(10)

wrote production/output/narrative_signals/topic_discovery_top_terms.csv
wrote production/output/narrative_signals/topic_discovery_growth.csv
wrote production/output/narrative_signals/P04_narrative_summary.md


,topic_id,early_share,late_share,growth,top_terms
7,7,0.028602,0.068655,0.040052,"pattern, runway, traffic, downwind, final, rad..."
14,14,0.029569,0.065077,0.035508,"door, ramp, gate, tug, push, brake, brakes, pa..."
15,15,0.034173,0.063833,0.029660,"runway, left, landing, plane, right, student, ..."
12,12,0.060605,0.072728,0.012124,"engine, oil, power, right engine, landing, lef..."
3,3,0.024622,0.036337,0.011715,"turbulence, wake, wake turbulence, encountered..."
1,1,0.035286,0.044681,0.009395,"smell, smoke, flight, odor, cabin, attendant, ..."
10,10,0.081355,0.087305,0.005950,"traffic, drone, feet, ra, ft, tcas, airspace, ..."
9,9,0.088210,0.089244,0.001034,"approach, visual, terrain, runway, altitude, i..."
6,6,0.178280,0.176875,-0.001406,"flight, maintenance, crew, mel, time, company,..."
0,0,0.027045,0.025480,-0.001565,"gear, landing gear, landing, nose, nose gear, ..."
